# Physics-Guided Motion Loss for Video Generation
**ArXivist-generated reproduction notebook**  
Paper: [arXiv 2506.02244v2](https://arxiv.org/abs/2506.02244)  
Generated: 2025-07-25  
SIR Confidence: 0.89

This notebook walks through every component of the physics-guided motion loss
pipeline — from raw video to scalar training signal — running fully on synthetic
data without any downloads.  Complete it top-to-bottom in under 30 minutes on
any machine with a modern GPU (or CPU with slower runtimes).

**Sections:**
1. Environment check
2. Installation
3. Paper overview
4. Spectral preprocessing (`SpectralProcessor`, `PolarLUT`)
5. Gate functions (`EnergyGate`, `ObservabilityGate`)
6. Translation loss (`TranslationalMotionLoss`)
7. Rotation loss (`RotationalMotionLoss`)
8. Scaling loss (`ScalingMotionLoss`)
9. Adaptive composite loss (`AdaptiveMotionLoss`)
10. Full pipeline (`PhysicsMotionLoss`)
11. Mini training demonstration
12. Paper results reference
13. Next steps


In [ ]:
import sys, torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM   : {vram:.1f} GB")
else:
    print("Running on CPU — demo cells will still work, training will be slow.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")


In [ ]:
import subprocess, sys
from pathlib import Path

repo_root = Path("..").resolve()   # notebooks/ → repo root
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(repo_root)],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Package installed in editable mode")
else:
    print("Installation output:")
    print(result.stdout[-2000:] if result.stdout else "")
    print(result.stderr[-2000:] if result.stderr else "")


In [ ]:
# Make sure the repo root is on sys.path for direct imports
import sys
from pathlib import Path

repo_root = str(Path("..").resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Repo root: {repo_root}")
print("Import path set ✓")


## 3. Paper Overview

### The Problem

Video diffusion models produce visually compelling frames, but the **motion between
frames is often physically wrong**: objects rubber-stretch, a spinning shoe reverses
direction mid-clip, a zooming train appears abruptly. Individual frames look fine —
the *temporal dynamics* do not.

### The Core Insight

Basic rigid motions leave **simple, exact fingerprints in the frequency domain**.
For a video with spatiotemporal spectrum $\hat{V}(\omega_x, \omega_y, \omega_t)$,
all three SIM(2) motion types concentrate energy on a single hyperplane:

$$
\omega_t + v_x \omega_x + v_y \omega_y + \Omega m + \alpha \nu + b_0 = 0
\quad \text{(Eq. 3.1)}
$$

where:
- $(v_x, v_y)$ = translation velocity → energy on a **plane** in $(\omega_x, \omega_y, \omega_t)$  
- $\Omega$ = angular velocity → energy on **tilted lines** in $(m, \omega_t)$  
- $\alpha = \dot{\sigma}$ = log-scale rate → energy on **tilted lines** in $(\nu, \omega_t)$

### The Solution

Add three differentiable frequency-domain losses to the backbone's denoising objective:

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{denoising}} + 0.1 \cdot \mathcal{L}_{\text{motion}}
$$

$$
\mathcal{L}_{\text{motion}} = \sum_{i} w_i L_i, \quad
w_i = \frac{\exp(-L_i/\tau)}{\sum_j \exp(-L_j/\tau)}
$$

**Key efficiency result (Section 4.1):** Only $\varrho^3 = 0.3^3 = 2.7\%$ of spectral
coefficients are processed, yet $\geq 97\%$ of the video's spectral energy is retained
(natural videos follow a $\omega^{-3.6}$ power law — energy concentrates at low frequencies).

### Implementation Map

| Paper Section | Code Module |
|--------------|-------------|
| Sec 4.1 — Low-pass truncation | `spectral/fft_utils.py → SpectralProcessor` |
| App A.6 — Gates | `spectral/gates.py → EnergyGate, ObservabilityGate` |
| Sec 3.4, App D.1 — $\mathcal{L}_{\text{trans}}$ | `losses/translation_loss.py` |
| Sec 3.5, App D.2 — $\mathcal{L}_{\text{rot}}$ | `losses/rotation_loss.py` |
| Sec 3.6 — $\mathcal{L}_{\text{scale}}$ | `losses/scaling_loss.py` |
| Sec 3.7 — Adaptive weighting | `losses/adaptive_composite.py` |
| Figure 1 — Full pipeline | `losses/physics_motion_loss.py` |


## 4. Synthetic Test Data

We create three canonical SIM(2) videos — one per motion type — to verify each
loss branch behaves as expected. All data is generated on-the-fly; no downloads needed.


In [ ]:
import torch
import torch.nn.functional as F

def make_translation_video(T=16, H=32, W=32, vx=1.0, vy=0.0):
    """Constant-velocity horizontal translation of a Gaussian blob."""
    cy, cx = H // 2, W // 2
    yy, xx = torch.meshgrid(
        torch.arange(H, dtype=torch.float32),
        torch.arange(W, dtype=torch.float32), indexing="ij"
    )
    blob = torch.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * (H/8)**2))
    frames = [torch.roll(blob, int(vx * t), dims=1) for t in range(T)]
    return torch.stack(frames)   # [T, H, W]

def make_rotation_video(T=16, H=32, W=32):
    """Uniform in-plane rotation of a Gaussian blob."""
    cy, cx = H // 2, W // 2
    yy, xx = torch.meshgrid(
        torch.arange(H, dtype=torch.float32),
        torch.arange(W, dtype=torch.float32), indexing="ij"
    )
    # Off-centre blob so rotation is visible
    blob = torch.exp(-((xx - cx - H//5)**2 + (yy - cy)**2) / (2 * (H/10)**2))
    blob = blob.unsqueeze(0).unsqueeze(0)  # [1,1,H,W] for grid_sample
    frames = []
    for t in range(T):
        angle_rad = (t / T) * 2 * torch.pi
        cos_a, sin_a = torch.cos(angle_rad), torch.sin(angle_rad)
        theta = torch.tensor([[cos_a, -sin_a, 0.0],
                               [sin_a,  cos_a, 0.0]], dtype=torch.float32).unsqueeze(0)
        grid = F.affine_grid(theta, blob.shape, align_corners=False)
        frames.append(F.grid_sample(blob, grid, align_corners=False).squeeze())
    return torch.stack(frames)   # [T, H, W]

def make_scaling_video(T=16, H=32, W=32):
    """Zoom-in: Gaussian blob grows over time (simulates uniform scaling)."""
    cy, cx = H // 2, W // 2
    yy, xx = torch.meshgrid(
        torch.arange(H, dtype=torch.float32),
        torch.arange(W, dtype=torch.float32), indexing="ij"
    )
    frames = []
    for t in range(T):
        sigma = (H / 16) * (0.5 + t / T)
        frame = torch.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
        frames.append(frame)
    return torch.stack(frames)   # [T, H, W]

# Generate all three
T, H, W = 16, 32, 32
v_trans  = make_translation_video(T, H, W)
v_rot    = make_rotation_video(T, H, W)
v_scale  = make_scaling_video(T, H, W)
v_random = torch.rand(T, H, W)

print(f"Translation video : {v_trans.shape} | min={v_trans.min():.2f} max={v_trans.max():.2f}")
print(f"Rotation video    : {v_rot.shape}   | min={v_rot.min():.2f} max={v_rot.max():.2f}")
print(f"Scaling video     : {v_scale.shape} | min={v_scale.min():.2f} max={v_scale.max():.2f}")
print(f"Random video      : {v_random.shape}")
print("\nSynthetic data ready ✓")


## 4. Spectral Preprocessing

### `SpectralProcessor` — 3D FFT + Low-Pass Cube

**Paper reference:** Appendix A.1, Section 4.1

The video window $V \in \mathbb{R}^{T \times H \times W}$ is transformed via:

1. **Hann window** $h[t]$ applied along time to reduce spectral leakage (App A.1)
2. **Separable 3D FFT**: 2D spatial DFT per frame, then 1D temporal DFT
3. **Low-pass cube truncation**: keep only the lowest $\varrho = 0.3$ fraction per dimension

$$
\eta_{\text{cube}}(0.3) \in [0.97, 0.987] \quad \text{(energy retained, App C)}
$$

This gives only $0.3^3 = 2.7\%$ of coefficients to process downstream.


In [ ]:
try:
    from src.physics_motion_loss.spectral.fft_utils import SpectralProcessor

    proc = SpectralProcessor(rho=0.3, Nr=12, M=16, Nxi=16)
    print(proc)

    # Full pipeline on translation video
    spectrum = proc.compute_spectrum(v_trans)      # [T, H, W] complex
    lp_cube  = proc.apply_lowpass_cube(spectrum)   # [T_lp, H_lp, W_lp]
    ring_E   = proc.get_ring_energies(lp_cube)     # [Nr, T_lp]
    polar    = proc.to_polar_sequence(lp_cube)     # [T_lp, Nr, M]

    T_lp, H_lp, W_lp = lp_cube.shape
    coeff_pct = (T_lp * H_lp * W_lp) / (T * H * W) * 100

    print(f"\nInput video    : [{T}, {H}, {W}]")
    print(f"Full spectrum  : {list(spectrum.shape)} complex  (dtype={spectrum.dtype})")
    print(f"Low-pass cube  : {list(lp_cube.shape)} → {coeff_pct:.1f}% of coefficients")
    print(f"Ring energies  : {list(ring_E.shape)}  [Nr rings × T_lp time steps]")
    print(f"Polar sequence : {list(polar.shape)} complex  [T_lp × Nr × M]")
    print(f"\nEnergy in lp cube vs full: "
          f"{(lp_cube.abs()**2).sum().item():.1f} / "
          f"{(spectrum.abs()**2).sum().item():.1f} = "
          f"{(lp_cube.abs()**2).sum().item()/(spectrum.abs()**2).sum().item()*100:.1f}%")

except Exception as e:
    print(f"Error: {e}")
    raise


## 5. Gate Functions

### `EnergyGate` and `ObservabilityGate` (Appendix A.6)

Before spectral samples enter the WLS regression, two soft gates down-weight
unreliable samples:

**Energy gate** — suppresses low-energy bins that are mostly noise:
$$g_E = \sigma\!\left(f \cdot \left(\frac{E}{E_{\max}} - \tau_E\right)\right), \quad \tau_E = 0.10,\ f = 10$$

**Observability gate** — suppresses angular harmonics near $m=0$ (DC carries no rotation info):
$$g_{\text{obs}}(m) = \frac{m^2}{m^2 + \lambda}, \quad \lambda = 1$$

Note $g_{\text{obs}}(0) = 0$ exactly — the $m=0$ harmonic is fully excluded from all rotation sums.


In [ ]:
try:
    from src.physics_motion_loss.spectral.gates import EnergyGate, ObservabilityGate

    # --- Energy gate ---
    gate_E = EnergyGate(tau_E=0.10, f=10.0)
    E_sample = torch.tensor([0.0, 0.05, 0.10, 0.20, 0.50, 1.0])
    E_max    = torch.tensor(1.0)
    weights_E = gate_E(E_sample, E_max)
    print("Energy gate response:")
    for e, w in zip(E_sample.tolist(), weights_E.tolist()):
        bar = "█" * int(w * 20)
        print(f"  E/E_max={e:.2f}  →  g_E={w:.3f}  {bar}")

    print()

    # --- Observability gate ---
    gate_obs = ObservabilityGate(lam=1.0)
    m_vals   = torch.tensor([-4., -3., -2., -1., 0., 1., 2., 3., 4.])
    weights_obs = gate_obs(m_vals)
    print("Observability gate g_obs(m) = m²/(m²+1):")
    for m, w in zip(m_vals.tolist(), weights_obs.tolist()):
        bar = "█" * int(w * 20)
        print(f"  m={int(m):+d}  →  g_obs={w:.3f}  {bar}")
    print()
    print("Note: m=0 is FULLY suppressed (g_obs=0.000) — DC carries no rotation info ✓")

except Exception as e:
    print(f"Error: {e}")
    raise


## 6. Translational Motion Loss $\mathcal{L}_{\text{trans}}$

**Paper reference:** Section 3.4, Appendix D.1

For constant-velocity translation $V(x,y,t) = V_0(x - v_x t,\; y - v_y t)$,
spectral energy concentrates on the plane:
$$\omega_t + v_x \omega_x + v_y \omega_y + b_0 = 0$$

The loss is the **normalised energy-weighted WLS residual** from fitting this plane:

$$\mathcal{L}_{\text{trans}} = \frac{\sum_i W_{ii}(A_i\hat{\beta} - b_i)^2}{\sum_i W_{ii}}$$

where rows $A_i = (\omega_{x,i},\; \omega_{y,i},\; 1)$, $b_i = -\omega_{t,i}$,
and $\hat{\beta} = [v_x, v_y, b_0]^\top$ solved via ridge WLS ($\lambda=10^{-3}$).

**Expected:** translation video → lower loss than random video.


In [ ]:
try:
    from src.physics_motion_loss.losses.translation_loss import TranslationalMotionLoss

    loss_trans_fn = TranslationalMotionLoss(ridge_lambda=1e-3)
    print(loss_trans_fn)

    results = {}
    for name, video in [
        ("Translation (ideal)", v_trans),
        ("Rotation",            v_rot),
        ("Scaling",             v_scale),
        ("Random",              v_random),
    ]:
        spec   = proc.compute_spectrum(video)
        lp     = proc.apply_lowpass_cube(spec)
        loss   = loss_trans_fn(lp).item()
        results[name] = loss

    print("\nL_trans values (lower = more translational):")
    max_loss = max(results.values())
    for name, loss in results.items():
        bar = "█" * int((loss / max_loss) * 30)
        print(f"  {name:25s}  {loss:.4f}  {bar}")

    t_loss = results["Translation (ideal)"]
    r_loss = results["Random"]
    status = "✓ PASS" if t_loss <= r_loss + 0.1 else "✗ FAIL"
    print(f"\n{status}: translation loss ({t_loss:.4f}) ≤ random loss ({r_loss:.4f})")

except Exception as e:
    print(f"Error: {e}")
    raise


## 7. Rotational Motion Loss $\mathcal{L}_{\text{rot}}$

**Paper reference:** Section 3.5, Appendix A.4, D.2

In-plane rotation $V(r,\theta,t) = V_0(r,\; \theta - \Omega t)$ produces two spectral signatures:

1. **Annular concentration** — energy rings around the origin  
2. **Tilted lines** in the $(m, \omega_t)$ plane: $\omega_t + m\Omega = 0$

The angular velocity estimate (Eq. A.7):
$$\Omega^* = -\frac{\sum_{\rho,m\neq0,\omega_t} |\tilde{C}_m|^2 \omega_t m}{\sum_{\rho,m\neq0,\omega_t} |\tilde{C}_m|^2 m^2}$$

The composite loss (Section 3.5):
$$\mathcal{L}_{\text{rot}} = 1 - \frac{C_{\text{ring}} + C_{\text{rot}}}{2}$$

where $C_{\text{ring}} = 1 - \bar{H}_{\text{ring}} / \log(N_r)$ (annular entropy)  
and $C_{\text{rot}} = E_{\text{line}} / E_{\text{all}}$ (tilted-line energy ratio).


In [ ]:
try:
    from src.physics_motion_loss.losses.rotation_loss import RotationalMotionLoss

    loss_rot_fn = RotationalMotionLoss(Nr=12, M=16, delta=1.0)
    print(loss_rot_fn)

    results = {}
    for name, video in [
        ("Rotation (ideal)", v_rot),
        ("Translation",      v_trans),
        ("Scaling",          v_scale),
        ("Random",           v_random),
    ]:
        spec   = proc.compute_spectrum(video)
        lp     = proc.apply_lowpass_cube(spec)
        ring_E = proc.get_ring_energies(lp)
        polar  = proc.to_polar_sequence(lp)
        loss   = loss_rot_fn(polar, ring_E).item()
        results[name] = loss

    print("\nL_rot values (lower = more rotational):")
    max_loss = max(results.values())
    for name, loss in results.items():
        bar = "█" * int((loss / max_loss) * 30)
        print(f"  {name:25s}  {loss:.4f}  {bar}")

    # Check ring concentration on rotation vs random
    spec_r = proc.compute_spectrum(v_rot)
    lp_r   = proc.apply_lowpass_cube(spec_r)
    ring_E_r = proc.get_ring_energies(lp_r)
    C_ring = loss_rot_fn._ring_concentration(ring_E_r).item()
    ring_E_rand = proc.get_ring_energies(proc.apply_lowpass_cube(proc.compute_spectrum(v_random)))
    C_ring_rand = loss_rot_fn._ring_concentration(ring_E_rand).item()
    print(f"\nC_ring (rotation video): {C_ring:.4f}  (higher = more annular concentration)")
    print(f"C_ring (random video)  : {C_ring_rand:.4f}")

except Exception as e:
    print(f"Error: {e}")
    raise


## 8. Scaling Motion Loss $\mathcal{L}_{\text{scale}}$

**Paper reference:** Section 3.6, Appendix A.4–A.5

Uniform scaling $V(x,y,t) = V_0(x/s(t),\; y/s(t))$ with $s(t) = e^{\sigma(t)}$ shifts
energy radially in the frequency domain. After a log-radius substitution $\xi = \log\rho$,
scaling becomes a temporal translation — concentrating on $\omega_t + \alpha\nu = 0$.

Two robust proxies (Section 3.6):

$$C_{\text{flow}} = \left|\sum_{k,t} \hat{\nabla}_\rho E_{k,t} \cdot \hat{\nabla}_t E_{k,t}\right|
\quad \text{(radial-temporal gradient alignment)}$$

$$S_{\text{trend}} = \left|\text{corr}(\rho_c, t)\right|, \quad
\rho_c(t) = \frac{\sum_k k \cdot E_k(t)}{\sum_k E_k(t) + \varepsilon}
\quad \text{(centroid trend)}$$

$$\mathcal{L}_{\text{scale}} = 1 - \frac{C_{\text{flow}} + S_{\text{trend}}}{2}$$

**Edge case (Section 3.6):** if $T_{\text{lp}} < 3$, both proxies default to 0.5.


In [ ]:
try:
    from src.physics_motion_loss.losses.scaling_loss import ScalingMotionLoss

    loss_scale_fn = ScalingMotionLoss(T_min=3)
    print(loss_scale_fn)

    results = {}
    for name, video in [
        ("Scaling (ideal)", v_scale),
        ("Translation",     v_trans),
        ("Rotation",        v_rot),
        ("Random",          v_random),
    ]:
        spec   = proc.compute_spectrum(video)
        lp     = proc.apply_lowpass_cube(spec)
        ring_E = proc.get_ring_energies(lp)
        loss   = loss_scale_fn(ring_E).item()
        results[name] = loss

    print("\nL_scale values (lower = more uniform-scaling):")
    max_loss = max(results.values())
    for name, loss in results.items():
        bar = "█" * int((loss / max_loss) * 30)
        print(f"  {name:25s}  {loss:.4f}  {bar}")

    # Demonstrate edge case: T_lp < 3 → 0.5
    ring_E_short = torch.rand(12, 2)   # T_lp = 2 < T_min = 3
    loss_short   = loss_scale_fn(ring_E_short).item()
    print(f"\nEdge case (T_lp=2 < T_min=3): L_scale = {loss_short:.4f} "
          f"{'✓ (should be 0.5)' if abs(loss_short - 0.5) < 1e-5 else '✗'}")

    # Show C_flow and S_trend for scaling video
    spec_s = proc.compute_spectrum(v_scale)
    lp_s   = proc.apply_lowpass_cube(spec_s)
    ring_E_s = proc.get_ring_energies(lp_s)
    C_flow   = loss_scale_fn._radial_flow_alignment(ring_E_s).item()
    S_trend  = loss_scale_fn._centroid_trend(ring_E_s).item()
    print(f"\nScaling video — C_flow={C_flow:.4f}, S_trend={S_trend:.4f}")
    print(f"  L_scale = 1 - ({C_flow:.4f} + {S_trend:.4f})/2 = {1-(C_flow+S_trend)/2:.4f}")

except Exception as e:
    print(f"Error: {e}")
    raise


## 9. Adaptive Composite Loss $\mathcal{L}_{\text{motion}}$

**Paper reference:** Section 3.7

The three losses are combined via softmax weighting grounded in the
**maximum-entropy principle** (Section 3.7):

$$w_i = \frac{\exp(-L_i / \tau)}{\sum_j \exp(-L_j / \tau)}, \quad \tau = 0.1$$

$$\mathcal{L}_{\text{motion}} = \sum_{i \in \{\text{trans, rot, scale}\}} w_i \cdot L_i$$

At $\tau = 0.1$ (nearly winner-takes-all), whichever loss is **smallest** (motion most
represented in the clip) receives the highest weight — automatically focusing training
on the dominant motion type without hard classification.


In [ ]:
try:
    from src.physics_motion_loss.losses.adaptive_composite import AdaptiveMotionLoss

    adaptive = AdaptiveMotionLoss(tau=0.1)
    print(adaptive)

    # Demonstrate temperature effect
    L_trans_ex = torch.tensor(0.20)   # low = translation-dominated clip
    L_rot_ex   = torch.tensor(0.70)
    L_scale_ex = torch.tensor(0.80)

    print("\nExample clip (translation-dominated):")
    print(f"  L_trans={L_trans_ex:.2f}, L_rot={L_rot_ex:.2f}, L_scale={L_scale_ex:.2f}")

    for tau in [0.01, 0.1, 1.0, 10.0]:
        af = AdaptiveMotionLoss(tau=tau)
        L_motion, w = af(L_trans_ex, L_rot_ex, L_scale_ex)
        print(f"  τ={tau:5.2f}  →  w=({w[0]:.2f}, {w[1]:.2f}, {w[2]:.2f})  "
              f"L_motion={L_motion.item():.4f}")

    print()
    print("At τ=0.01 (winner-takes-all): translation dominates entirely.")
    print("At τ=10.0 (uniform):          all three contribute equally.")

    # Verify weights sum to 1
    _, w = adaptive(L_trans_ex, L_rot_ex, L_scale_ex)
    print(f"\nWeights sum: {w.sum().item():.6f}  {'✓' if abs(w.sum().item()-1) < 1e-5 else '✗'}")

except Exception as e:
    print(f"Error: {e}")
    raise


## 10. Full Pipeline — `PhysicsMotionLoss`

**Paper reference:** Figure 1, Section 3, Section 4.1

The orchestrator processes `x̂₀ ∈ ℝ^{B × C × T × H × W}` — the denoised prediction
at each diffusion timestep — and returns a scalar $\mathcal{L}_{\text{motion}}$
that is added to the backbone denoising loss with weight 0.1.

All spectral/solver blocks run in FP32 via `FP32Context`, even when the backbone
uses BF16. RGB channels are processed independently with energies summed (App A.1).

The full data flow (Figure 1):
```
x̂₀ → [per channel] → 3D FFT + Hann → low-pass cube (2.7% coeffs)
    → L_trans (WLS plane)  + L_rot (ring/tilted-line) + L_scale (grad/centroid)
    → adaptive softmax weighting (τ=0.1)  →  L_motion (scalar)
```


In [ ]:
try:
    from src.physics_motion_loss import PhysicsMotionLoss

    loss_fn = PhysicsMotionLoss(rho=0.3, Nr=12, M=16)
    print(loss_fn)
    print()

    # Test all four video types as [B=1, C=1, T, H, W]
    print("Full pipeline output on each video type:")
    print(f"{'Video':25s}  {'L_motion':>8}  {'L_trans':>8}  {'L_rot':>8}  "
          f"{'L_scale':>8}  {'w_t':>5}  {'w_r':>5}  {'w_s':>5}")
    print("─" * 85)

    for name, video in [
        ("Translation", v_trans),
        ("Rotation",    v_rot),
        ("Scaling",     v_scale),
        ("Random",      v_random),
    ]:
        x = video.unsqueeze(0).unsqueeze(0)   # [1, 1, T, H, W]
        out = loss_fn(x.float())
        print(f"  {name:23s}  "
              f"{out['loss'].item():8.4f}  "
              f"{out['L_trans'].item():8.4f}  "
              f"{out['L_rot'].item():8.4f}  "
              f"{out['L_scale'].item():8.4f}  "
              f"{out['w_trans'].item():5.2f}  "
              f"{out['w_rot'].item():5.2f}  "
              f"{out['w_scale'].item():5.2f}")

    # RGB batch test
    x_rgb = torch.rand(2, 3, T, H, W)
    out_rgb = loss_fn(x_rgb)
    print(f"\nRGB batch [B=2, C=3, T={T}, H={H}, W={W}] → L_motion={out_rgb['loss'].item():.4f} ✓")

    # Gradient flow test
    x_grad = torch.rand(1, 1, T, H, W, requires_grad=True)
    out_grad = loss_fn(x_grad)
    out_grad['loss'].backward()
    has_grad = x_grad.grad is not None and not torch.isnan(x_grad.grad).any()
    print(f"Gradient flows back to input: {'✓' if has_grad else '✗'}")

except Exception as e:
    print(f"Error: {e}")
    raise


## 11. Mini Training Demonstration

This cell demonstrates the physics loss added to a **mock denoising objective**
on synthetic data. We use a tiny linear backbone (not a real diffusion model)
purely to confirm:

1. The loss computation is differentiable end-to-end  
2. The combined loss decreases over training steps  
3. The adaptive weights evolve as training progresses

> **Paper setup (Section 4.1):** Full training uses 4 epochs, cosine LR from 2e-5,
> on 4× A100 GPUs with the Open-Sora / MVDIT / Hunyuan backbones.  
> This demo uses a tiny linear model and 20 steps to verify the pipeline.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ── Tiny mock backbone ────────────────────────────────────────────────────────
# In real training this would be Open-Sora / MVDIT / Hunyuan.
# Here we use a small linear layer purely to test the gradient pipeline.

B, C_in, T_demo, H_demo, W_demo = 2, 1, 8, 16, 16

class MockDiffusionBackbone(nn.Module):
    """Minimal stand-in for a real diffusion backbone (for demo only)."""
    def __init__(self):
        super().__init__()
        dim = C_in * T_demo * H_demo * W_demo
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(dim, dim // 4),
            nn.ReLU(),
            nn.Linear(dim // 4, dim),
        )
        self.unflatten = nn.Unflatten(1, (C_in, T_demo, H_demo, W_demo))

    def forward(self, x_noisy, t=None):
        # Predict x0_hat; mock denoising loss = MSE vs input
        x0_hat = self.unflatten(self.net(x_noisy))
        denoising_loss = ((x0_hat - x_noisy) ** 2).mean()
        return {"loss": denoising_loss, "x0_hat": x0_hat.detach()}

model    = MockDiffusionBackbone().to(device)
loss_fn  = PhysicsMotionLoss(rho=0.3, Nr=8, M=12).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)   # ASSUMED AdamW (conf 0.65)
PHYSICS_WEIGHT = 0.1   # paper Table 5

# ── Synthetic noisy video dataset ─────────────────────────────────────────────
def sample_batch(B, C, T, H, W, device):
    # Mix of motion types for a realistic synthetic batch
    videos = []
    for _ in range(B):
        kind = torch.randint(0, 3, (1,)).item()
        if kind == 0:
            v = make_translation_video(T, H, W, vx=float(torch.rand(1)*2))
        elif kind == 1:
            v = make_scaling_video(T, H, W)
        else:
            v = torch.rand(T, H, W)
        videos.append(v.unsqueeze(0))  # [1, T, H, W]
    return torch.stack(videos, dim=0).to(device)  # [B, 1, T, H, W]

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Step':>4}  {'Denoise':>9}  {'L_motion':>9}  {'Total':>9}  "
      f"{'w_trans':>7}  {'w_rot':>7}  {'w_scale':>7}")
print("─" * 70)

for step in range(20):
    optimizer.zero_grad()

    x_noisy = sample_batch(B, C_in, T_demo, H_demo, W_demo, device)
    noise   = torch.randn_like(x_noisy) * 0.1
    x_in    = (x_noisy + noise).clamp(0, 1)

    # Backbone forward
    model_out      = model(x_in)
    denoising_loss = model_out["loss"]
    x0_hat         = model_out["x0_hat"]

    # Physics loss (FP32Context enforced inside)
    phys_out = loss_fn(x0_hat.float())
    L_motion = phys_out["loss"]

    total_loss = denoising_loss + PHYSICS_WEIGHT * L_motion
    total_loss.backward()
    optimizer.step()

    if step % 4 == 0 or step == 19:
        print(f"  {step:2d}    {denoising_loss.item():9.5f}  "
              f"{L_motion.item():9.5f}  {total_loss.item():9.5f}  "
              f"{phys_out['w_trans'].item():7.3f}  "
              f"{phys_out['w_rot'].item():7.3f}  "
              f"{phys_out['w_scale'].item():7.3f}")

print("\nMini training complete ✓")
print("Gradients flowed through both denoising and physics loss at every step.")


## 12. Paper Results Reference

Results reported in the paper (Table 1a, 1b, 2 of arXiv 2506.02244v2).
These are the targets to match when running full training on OpenVID-1M.


In [ ]:
# Results from SIR evaluation_protocol.primary_results
# Paper: arXiv 2506.02244v2, Tables 1a, 1b, 2

paper_results = {
    "dataset": "OpenVID-1M (EvalCrafter protocol)",
    "Open-Sora": {
        "Action Recognition Score": {"baseline": 60.77, "ours": 69.71, "delta": "+14.7%"},
        "Motion Accuracy Score":    {"baseline": 44.00, "ours": 49.00, "delta": "+11.4%"},
        "Warping Error":            {"baseline": 0.0089,"ours": 0.0056,"delta": "-37.1%"},
        "Text-Video Alignment":     {"baseline": 54.02, "ours": 61.05, "delta": "+13.0%"},
        "CLIP Temporal Score":      {"baseline": 99.80, "ours": 99.85, "delta": "+0.05"},
    },
    "MVDIT": {
        "Action Recognition Score": {"baseline": 62.34, "ours": 69.70, "delta": "+11.8%"},
        "Motion Accuracy Score":    {"baseline": 44.00, "ours": 51.00, "delta": "+15.9%"},
        "Warping Error":            {"baseline": 0.0080,"ours": 0.0062,"delta": "-22.5%"},
        "Flow Score":               {"baseline": 1.01,  "ours": 1.22,  "delta": "+20.8%"},
    },
    "Hunyuan (LoRA)": {
        "Action Recognition Score": {"baseline": 68.93, "ours": 73.15, "delta": "+6.1%"},
        "Motion Accuracy Score":    {"baseline": 56.00, "ours": 59.00, "delta": "+5.4%"},
        "Warping Error":            {"baseline": 0.0024,"ours": 0.0016,"delta": "-33.3%"},
        "Text-Video Alignment":     {"baseline": 59.60, "ours": 65.34, "delta": "+9.6%"},
    },
    "User study preference (2AFC)": {
        "Open-Sora (quality/motion)":  "79.4%",
        "Open-Sora (text alignment)":  "74.2%",
        "MVDIT (quality/motion)":      "82.7%",
        "MVDIT (text alignment)":      "77.9%",
    }
}

print("=" * 65)
print("Paper's Reported Results — arXiv 2506.02244v2")
print("=" * 65)
for backbone, metrics in paper_results.items():
    if backbone == "dataset":
        print(f"Dataset: {metrics}\n")
        continue
    print(f"\n── {backbone} ──")
    if isinstance(metrics, dict):
        for metric, vals in metrics.items():
            if isinstance(vals, dict):
                print(f"  {metric:35s}: "
                      f"baseline={vals['baseline']}  →  ours={vals['ours']}  "
                      f"({vals['delta']})")
            else:
                print(f"  {metric:35s}: {vals}")

print("\n" + "=" * 65)
print("To reproduce these results:")
print("  1. python train.py --config configs/config.yaml --backbone open_sora")
print("     --data_root /path/to/openvid1m --output_dir outputs/open_sora_physics")
print("  2. python evaluate.py --checkpoint outputs/open_sora_physics/checkpoint_best.pt")
print("     --backbone open_sora --data_root /path/to/openvid1m --output_dir outputs/eval")
print("  3. Feed your results into Stage 6 (Results Comparator) for automated comparison.")


## 13. Verify Test Suite

Run the full test suite to confirm the installation is correct before starting
full-scale training.


In [ ]:
import subprocess, sys
from pathlib import Path

repo_root = Path("..").resolve()
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(repo_root / "tests"), "-v", "--tb=short", "-q"],
    capture_output=True, text=True, cwd=str(repo_root)
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode == 0:
    print("\n✓ All tests passed — ready for full training.")
else:
    print("\n✗ Some tests failed — check the output above before proceeding.")


## 14. What to Do Next

### Full Training
```bash
# Open-Sora backbone (paper's primary backbone)
python train.py \
    --config configs/config.yaml \
    --backbone open_sora \
    --data_root /path/to/openvid1m \
    --output_dir outputs/open_sora_physics

# Hunyuan with LoRA (WARNING: LoRA rank=16 is assumed — ablate over {4,8,16,32})
python train.py --config configs/config.yaml --backbone hunyuan \
    --use_lora --data_root /path/to/openvid1m --output_dir outputs/hunyuan_lora
```

### Ablation Study (Table 4)
```bash
for mode in full no_trans no_rot no_scale flow_only; do
    python scripts/ablation.py --config configs/config.yaml \
        --backbone open_sora --data_root /data \
        --output_dir outputs/ablation --ablation_mode $mode
done
```

### Spectral Debugging
```bash
# Visualise the full spectral pipeline on a real video
python scripts/visualize_spectrum.py \
    --video /path/to/video.mp4 \
    --output_dir outputs/spectral_debug
```

---

### Key Implementation Assumptions (from SIR)

| Assumption | Confidence | Risk |
|------------|-----------|------|
| **Optimizer: AdamW** — not named in paper | 0.65 | 🔴 High — test Adam/Lion |
| **LoRA rank=16** — not specified | <0.60 | 🔴 High — sweep {4,8,16,32} |
| **Physics weight constant=0.1** | 0.78 | 🟡 Medium — try cosine-annealed |
| **Temporal window T=16** (range 12–16) | 0.72 | 🟡 Medium — test T=12 |
| **Stop-grad on adaptive weights: False** | 0.70 | 🟡 Medium — test True |

---

### ArXivist Pipeline Status

| Stage | Status |
|-------|--------|
| 1 — Paper Parser | ✅ Complete |
| 2 — SIR Registry | ✅ Complete |
| 3 — Architecture Planner | ✅ Complete |
| 4 — Code Generator | ✅ Complete (44/44 tests passing) |
| 5 — Notebook Generator | ✅ Complete |
| 6 — Results Comparator | ⏳ Pending |

Say **"move to stage 6"** to run the Results Comparator once you have training outputs.
